# updt.pro CV — cut a golden clip + GPU sanity run

Takes your **full match** (`match1.mp4`, kept in Google Drive), snips a short **2-minute golden clip** from it, and runs the perception + tagging pipeline on that clip on the Colab **GPU**. The short clip is what you'll hand-annotate for the golden set (plan §10.1).

**Before you run:**
1. `Runtime -> Change runtime type -> GPU` (T4 is enough).
2. Upload `match1.mp4` to your **Google Drive** (drive.google.com). This notebook reads it from there.
3. `Runtime -> Run all`. Cell 2 asks you to upload `updtvision_showcase.zip` (the code).


In [ ]:
# 0) Confirm a GPU is attached
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime>GPU')

In [ ]:
# 1) Load the code from the uploaded zip. This cell handles both ways of getting it in:
#    (a) you already unzipped it via the Files panel -> it just finds and cds into it;
#    (b) otherwise it opens an upload dialog -> pick updtvision_showcase.zip, it unzips.
import os, glob, zipfile

def _find_root():
    # a project root is any dir that contains both run.py and src/
    for p in glob.glob('/content/**/run.py', recursive=True):
        d = os.path.dirname(p)
        if os.path.isdir(os.path.join(d, 'src')):
            return d
    return None

root = _find_root()
if root is None:
    from google.colab import files
    print('Pick updtvision_showcase.zip ...')
    up = files.upload()
    for name in up:
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall('/content/code')
    root = _find_root()
assert root, 'No run.py + src/ found - upload updtvision_showcase.zip and re-run this cell.'
os.chdir(root)              # absolute chdir: survives re-runs / runtime restarts
print('cwd:', os.getcwd())
print(sorted(os.listdir('.')))

In [ ]:
# 2) Install deps. Keep Colab's preinstalled numpy (2.x) to AVOID a runtime restart
#    (our requirements pin numpy<2 for local arm64; on Colab numpy 2 is fine).
!grep -ivE '^numpy' requirements.txt > /tmp/reqs_colab.txt
!pip install -q -r /tmp/reqs_colab.txt
print('deps installed (numpy kept at Colab default)')

In [ ]:
# 3) Download pretrained weights (players / ball / pitch). No Roboflow API key needed.
#    We only need the weights here - the match video comes from your Drive, not the sample clip.
!python scripts/download_assets.py
!ls -lh data/models

## Mount Drive and choose which 2 minutes to cut

Edit the three settings in the next cell if needed:
- **`DRIVE_VIDEO`** - where your match lives in Drive. Default assumes you dropped `match1.mp4` in the top level (`My Drive`).
- **`CLIP_START`** - where the clip begins, as `HH:MM:SS`. Pick a passage of normal open play with the pitch clearly visible (avoid replays / close-ups). `00:20:00` = 20 minutes in.
- **`CLIP_SECONDS`** - how long the clip is. 120 = 2 minutes (a good, annotatable size).


In [ ]:
# 4) Mount your Google Drive and point at the match video
from google.colab import drive
drive.mount('/content/drive')

# ---- EDIT THESE THREE IF NEEDED ----
DRIVE_VIDEO  = '/content/drive/MyDrive/match1.mp4'   # where your match is in Drive
CLIP_START   = '00:20:00'                            # HH:MM:SS - where the golden clip begins
CLIP_SECONDS = 120                                   # clip length in seconds (120 = 2 min)
# ------------------------------------

import os
assert os.path.exists(DRIVE_VIDEO), (
    f'Not found: {DRIVE_VIDEO}\n'
    'Check the path. Tip: open the Files panel (folder icon, left) -> drive -> MyDrive '
    'and confirm the exact filename, then fix DRIVE_VIDEO above.'
)
print('found match video:', DRIVE_VIDEO, f'({os.path.getsize(DRIVE_VIDEO)/1e9:.1f} GB)')

In [ ]:
# 5) Cut the short golden clip with ffmpeg (re-encoded so OpenCV reads it cleanly).
#    Output goes to data/raw_matches/ - the folder the pipeline reads from.
CLIP_PATH = 'data/raw_matches/match1_gs.mp4'
!mkdir -p data/raw_matches
!ffmpeg -y -ss {CLIP_START} -i "{DRIVE_VIDEO}" -t {CLIP_SECONDS} \
    -c:v libx264 -preset veryfast -crf 20 -an {CLIP_PATH}
!ls -lh {CLIP_PATH}

# sanity: confirm the clip's length + fps via the project's own reader
from src.ingestion.video import get_video_info
i = get_video_info(CLIP_PATH)
print(f'clip: {i.width}x{i.height}  {i.fps:.0f} fps  {i.total_frames} frames  ({i.total_frames/i.fps:.0f} s)')

In [ ]:
# 6) Run the full pipeline on the golden clip, on GPU. ~2 min of footage -> a few minutes on a T4.
#    First run downloads SigLIP (~1.5 GB) for team classification - one-time.
!PYTORCH_ENABLE_MPS_FALLBACK=1 python run.py \
    --source data/raw_matches/match1_gs.mp4 \
    --device cuda \
    --output-dir outputs/match1_gs

## See the tracking - render the overlay video

`run.py` above produced the event *data*. This step draws the perception on the clip: team-coloured **tracks + IDs**, goalkeeper/referee flags, the **ball**, a HUD (clock, calibration bar, ball status) and the live event log. `--only showcase` renders just this tracking view (skips the tactical maps), so it's quicker.

It renders **every frame** for smooth playback (~2-min clip = a few minutes on a T4). For a quick look, add `--max-frames 900` to render just the first 30 seconds.


In [ ]:
# 7) Render the tracking overlay onto the clip, on GPU.
!PYTORCH_ENABLE_MPS_FALLBACK=1 python scripts/render_video.py \
    --source data/raw_matches/match1_gs.mp4 \
    --device cuda --only showcase \
    --output-dir outputs/showcase

# Transcode to browser/QuickTime-friendly H.264 so it plays anywhere.
import subprocess
subprocess.run(['ffmpeg','-y','-i','outputs/showcase/showcase.mp4',
    '-vcodec','libx264','-pix_fmt','yuv420p','-movflags','+faststart',
    'outputs/showcase/showcase_h264.mp4'], check=True)
print('rendered outputs/showcase/showcase_h264.mp4')

In [ ]:
# 8) Preview the tracking video inline
import base64
from IPython.display import HTML, display
data = base64.b64encode(open('outputs/showcase/showcase_h264.mp4','rb').read()).decode()
display(HTML(f'<video width=900 controls src="data:video/mp4;base64,{data}"></video>'))

In [ ]:
# 9) Inspect the run: summary numbers + the event log the analyst/DB will consume
import json, pandas as pd
print(json.dumps(json.load(open('outputs/match1_gs/run_summary.json')), indent=2))
events = json.load(open('outputs/match1_gs/events.json'))
df = pd.DataFrame(events)
print(f'\n{len(df)} events')
if len(df):
    print(df['type'].value_counts().to_dict())
    cols = ['type','team_id','start_ts','end_ts','length_m','confidence','review_status']
    display(df[[c for c in cols if c in df.columns]].head(25))

## Download the clip + its events (for annotation)

The next cell downloads `match1_gs.mp4` and its `events.json`. Keep both:
- upload **`match1_gs.mp4`** to Roboflow to hand-annotate the ground truth, and
- the pipeline's **`events.json`** is the machine's answer, which you'll score *against* your annotations with `scripts/evaluate.py`.

This is the clip that becomes your first golden-set entry (`data/golden_set/gs_match1_0/`).


In [ ]:
# 10) Download the clip + events (+ the tracking video) for local annotation
import os, zipfile
out = '/content/match1_gs_bundle.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('data/raw_matches/match1_gs.mp4', 'match1_gs.mp4')
    z.write('outputs/match1_gs/events.json', 'events.json')
    z.write('outputs/match1_gs/run_summary.json', 'run_summary.json')
    if os.path.exists('outputs/showcase/showcase_h264.mp4'):
        z.write('outputs/showcase/showcase_h264.mp4', 'showcase_tracking.mp4')
from google.colab import files
files.download(out)